# Model Checkpoint Evaluation

This notebook allows you to:
1. Select a specific model checkpoint from a trained experiment
2. Choose between test/val splits for evaluation
3. Compute segment-level and TIC-level metrics
4. Generate confusion matrices


In [1]:
import os
import json
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

from trainer.config import ExperimentConfig, load_config
from trainer.dataset import (
    LightCurveNPZDataset,
    LightCurveCollator,
    collect_files_labels_and_groups,
    group_stratified_split,
)
from trainer.model import MultiBranchClassifier
from trainer.evaluation import evaluate_dataset_segment_and_tic, predict_dataset, compute_segment_metrics, compute_tic_mean_prob_metrics, get_tic_prediction_pairs
from trainer.utils import enabled_flux_branches, get_patch_size, set_reproducibility

# Set up matplotlib
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("Imports successful!")

Imports successful!


## Step 1: Discover Available Experiments and Checkpoints

In [2]:
# List all available experiment directories
results_dir = Path("phase3/results")
experiment_dirs = sorted([d for d in results_dir.glob("*") if d.is_dir()])

print("Available experiments:")
for i, exp_dir in enumerate(experiment_dirs, 1):
    print(f"{i}. {exp_dir.name}")

# Allow user to select experiment
selected_exp_idx = 3
selected_exp_dir = experiment_dirs[selected_exp_idx - 1]
print(f"\nSelected experiment: {selected_exp_dir.name}")

# Get the actual experiment folder (usually numbered)
actual_exp_dirs = sorted([d for d in selected_exp_dir.glob("*") if d.is_dir()])
if len(actual_exp_dirs) == 1:
    experiment_path = actual_exp_dirs[0]
else:
    experiment_path = selected_exp_dir

print(f"Experiment path: {experiment_path}")

Available experiments:
1. A_chronos_callall
2. A_qwen_call
3. B_chronos_callall
4. B_chronos_callall2
5. B_qwen_call

Selected experiment: B_chronos_callall
Experiment path: phase3/results/B_chronos_callall/001_B_chronos_callall


## Step 2: List and Select a Checkpoint

In [3]:
# Find all checkpoints
checkpoint_dirs = sorted(
    [d for d in experiment_path.glob("checkpoint-*") if d.is_dir()],
    key=lambda x: int(x.name.split("-")[1])
)

print(f"Found {len(checkpoint_dirs)} checkpoints:")
for i, ckpt_dir in enumerate(checkpoint_dirs[:10], 1):
    print(f"{i}. {ckpt_dir.name}")
if len(checkpoint_dirs) > 10:
    print(f"... and {len(checkpoint_dirs) - 10} more")

# Also list pre-trained models
best_model_path = experiment_path / "best_model.pt"
best_tic_model_path = experiment_path / "best_tic_model.pt"

print(f"\nSpecial models:")
if best_model_path.exists():
    print(f"  - best_model.pt (exists)")
if best_tic_model_path.exists():
    print(f"  - best_tic_model.pt (exists)")

# Select checkpoint: either a specific checkpoint or best model
use_best_tic_model = True
checkpoint_idx = 30

if use_best_tic_model and best_tic_model_path.exists():
    checkpoint_path = experiment_path
    model_file = best_tic_model_path
    print(f"\nUsing: {model_file.name}")
elif checkpoint_idx < len(checkpoint_dirs):
    checkpoint_path = checkpoint_dirs[checkpoint_idx]
    model_file = checkpoint_path / "model.safetensors"
    print(f"\nUsing checkpoint: {checkpoint_path.name}")
else:
    raise ValueError(f"Invalid checkpoint index: {checkpoint_idx}")

Found 10 checkpoints:
1. checkpoint-335
2. checkpoint-670
3. checkpoint-1005
4. checkpoint-1340
5. checkpoint-1675
6. checkpoint-2010
7. checkpoint-2345
8. checkpoint-2680
9. checkpoint-3015
10. checkpoint-3350

Special models:
  - best_model.pt (exists)
  - best_tic_model.pt (exists)

Using: best_tic_model.pt


## Step 3: Load Configuration and Set Up Dataset

In [4]:
# Load the resolved config
config_path = experiment_path / "resolved_config.json"
cfg = load_config(config_path)

print(f"Config loaded from: {config_path}")
print(f"Data directory: {cfg.DATA_DIR}")
print(f"Allowed classes: {cfg.ALLOWED_CLASSES}")
print(f"Enabled branches: {enabled_flux_branches(cfg)}")

# Set up device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")

# Set reproducibility
set_reproducibility(cfg)

Config loaded from: phase3/results/B_chronos_callall/001_B_chronos_callall/resolved_config.json
Data directory: ../../data/processed_4_channel_multi_256
Allowed classes: ['pulsating', 'eclipsing', 'rotating']
Enabled branches: ['2min', '8min', '32min', '128min']

Device: cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [5]:
# Collect data files
all_files, all_label_strings, y_all, groups_all, segments_all, n_segments_all, label2id, id2label = collect_files_labels_and_groups(
    cfg, cfg.DATA_DIR, cfg.ALLOWED_CLASSES
)

print(f"Total files: {len(all_files)}")
print(f"Label mapping: {label2id}")
print(f"Class names: {[id2label[i] for i in range(len(id2label))]}")

Found total NPZ files: 30850
Kept files: 30850
Unique TIC IDs kept: 898
Enabled flux branches: ['2min', '8min', '32min', '128min']
USE_PHASE_BRANCH: True
USE_EXTRA_FEATURES_BRANCH: True
n_segments summary: min=2, median=47.0, max=52
Total files: 30850
Label mapping: {'eclipsing': 0, 'pulsating': 1, 'rotating': 2}
Class names: ['eclipsing', 'pulsating', 'rotating']


In [6]:
# Split data the same way as training
train_files, val_files, test_files, train_labels_str, val_labels_str, test_labels_str = group_stratified_split(
    file_paths=all_files,
    labels_str=all_label_strings,
    groups=groups_all,
    val_size=cfg.VAL_SIZE,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.RANDOM_STATE,
)

print(f"Train files: {len(train_files)}")
print(f"Val files: {len(val_files)}")
print(f"Test files: {len(test_files)}")

Train files: 21398
Val files: 4800
Test files: 4652


In [7]:
# Select which split to evaluate
eval_split = "test"  # @param ["val", "test"] {type: "string"}

if eval_split == "val":
    eval_files = val_files
    eval_labels = val_labels_str
elif eval_split == "test":
    eval_files = test_files
    eval_labels = test_labels_str
else:
    raise ValueError(f"Unknown split: {eval_split}")

print(f"Evaluating on {eval_split} split")
print(f"Number of files: {len(eval_files)}")

# Create dataset
eval_ds = LightCurveNPZDataset(cfg, eval_files, label2id)
print(f"Dataset created with {len(eval_ds)} samples")

Evaluating on test split
Number of files: 4652
Loaded 1000 records into memory...
Loaded 2000 records into memory...
Loaded 3000 records into memory...
Loaded 4000 records into memory...
Loaded 4652 records into memory.
Dataset created with 4652 samples


## Step 4: Load Model and Checkpoint

In [8]:
# Build model
branch_input_dim = 1 + int(cfg.USE_FLUX_ERR)
extra_input_dim = 1 + 10 + 10 if cfg.USE_EXTRA_FEATURES_BRANCH else 0
n_classes = len([id2label[i] for i in range(len(id2label))])

model = MultiBranchClassifier(
    cfg=cfg,
    n_classes=n_classes,
    branch_input_dim=branch_input_dim,
    d_model=cfg.D_MODEL,
    n_heads=cfg.N_HEADS,
    n_layers=cfg.N_LAYERS,
    ff_dim=cfg.FF_DIM,
    dropout=cfg.DROPOUT,
    extra_input_dim=extra_input_dim,
    extra_hidden_dim=cfg.EXTRA_MLP_HIDDEN,
    extra_out_dim=cfg.EXTRA_MLP_OUT,
).to(device)

print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

Model created with 604,331,171 parameters


In [9]:
# Load checkpoint
from safetensors.torch import load_file

if model_file.suffix == ".safetensors":
    state_dict = load_file(model_file)
    model.load_state_dict(state_dict)
    print(f"Loaded safetensors checkpoint: {model_file}")
elif model_file.suffix == ".pt":
    checkpoint = torch.load(model_file, map_location=device)

    # Handle wrapped checkpoints (from trainers that save full state)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=False)
    print(f"Loaded torch checkpoint: {model_file}")
else:
    raise ValueError(f"Unsupported model format: {model_file}")

model.eval()
print("Model loaded and set to evaluation mode")

Loaded torch checkpoint: phase3/results/B_chronos_callall/001_B_chronos_callall/best_tic_model.pt
Model loaded and set to evaluation mode


## Step 5: Run Inference on Evaluation Split

In [10]:
# Create collator
collator = LightCurveCollator(cfg)

# Run predictions
print(f"Running inference on {eval_split} split...")
pred = predict_dataset(
    cfg=cfg,
    model=model,
    dataset=eval_ds,
    batch_size=cfg.BATCH_SIZE,
    device=device,
    collator=collator,
)

print(f"Predictions completed")
print(f"  Logits shape: {pred['logits'].shape}")
print(f"  Labels shape: {pred['labels'].shape}")
print(f"  TIC IDs: {len(np.unique(pred['tic_ids']))} unique")

Running inference on test split...
Predictions completed
  Logits shape: (4652, 3)
  Labels shape: (4652,)
  TIC IDs: 135 unique


In [11]:
preds = np.argmax(pred['logits'], axis=1)

In [12]:
pairs = get_tic_prediction_pairs(pred['logits'], pred['labels'], pred['tic_ids'], id2label)

In [13]:
print(pairs)

{21002602: ('rotating', 'rotating'), 21162252: ('pulsating', 'pulsating'), 47670480: ('rotating', 'rotating'), 160518449: ('eclipsing', 'eclipsing'), 160533144: ('pulsating', 'pulsating'), 160618495: ('pulsating', 'pulsating'), 165305498: ('pulsating', 'pulsating'), 198211758: ('rotating', 'rotating'), 198383073: ('pulsating', 'pulsating'), 198410480: ('eclipsing', 'eclipsing'), 198416255: ('pulsating', 'pulsating'), 199687298: ('pulsating', 'pulsating'), 207438428: ('pulsating', 'pulsating'), 219740926: ('rotating', 'rotating'), 219755088: ('pulsating', 'pulsating'), 219772324: ('pulsating', 'pulsating'), 219810107: ('rotating', 'rotating'), 219857577: ('rotating', 'rotating'), 219857708: ('pulsating', 'pulsating'), 219861551: ('pulsating', 'pulsating'), 219869224: ('pulsating', 'pulsating'), 219893761: ('pulsating', 'pulsating'), 219894715: ('rotating', 'rotating'), 224595426: ('pulsating', 'eclipsing'), 229440890: ('rotating', 'rotating'), 229452839: ('rotating', 'rotating'), 229598

## Step 6: Compute Segment and TIC Metrics

In [ ]:
# Compute segment-level metrics
print("\n" + "="*60)
print("SEGMENT-LEVEL METRICS")
print("="*60)

segment_metrics = compute_segment_metrics(pred["logits"], pred["labels"])
for k, v in segment_metrics.items():
    print(f"{k}: {v:.6f}")

# Save segment metrics
segment_metrics_dict = {k: float(v) for k, v in segment_metrics.items()}
print(f"\nSegment metrics: {segment_metrics_dict}")

In [ ]:
# Compute TIC-level metrics
print("\n" + "="*60)
print("TIC-LEVEL METRICS (Mean Probability Aggregation)")
print("="*60)

tic_metrics, tic_agg = compute_tic_mean_prob_metrics(
    logits=pred["logits"],
    labels=pred["labels"],
    tic_ids=pred["tic_ids"],
)

for k, v in tic_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.6f}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Combined metrics summary
print("\n" + "="*60)
print(f"{eval_split.upper()} SPLIT - COMBINED METRICS SUMMARY")
print("="*60)

combined_metrics = {**segment_metrics, **tic_metrics}
metrics_df = pd.DataFrame([
    {"metric": k, "value": v if isinstance(v, (int, float)) else str(v)}
    for k, v in combined_metrics.items()
])
print(metrics_df.to_string(index=False))

## Step 7: Compute Confusion Matrices

In [ ]:
# Get class names
class_names = [id2label[i] for i in range(len(id2label))]
print(f"Class names: {class_names}")

# Segment-level confusion matrix
segment_preds = np.argmax(pred["logits"], axis=1)
segment_cm = confusion_matrix(
    pred["labels"], segment_preds, labels=np.arange(len(class_names))
)

print("\nSegment-level Confusion Matrix:")
print(segment_cm)

In [ ]:
# TIC-level confusion matrix
tic_cm = confusion_matrix(
    tic_agg["tic_true"], tic_agg["tic_pred"], labels=np.arange(len(class_names))
)

print("TIC-level Confusion Matrix:")
print(tic_cm)

## Step 8: Visualize Confusion Matrices

In [ ]:
# Plot segment-level confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=segment_cm, display_labels=class_names
)
disp.plot(ax=ax, values_format="d", xticks_rotation=45, cmap="viridis")
ax.set_title(f"{eval_split.upper()} Split - Segment-level Confusion Matrix", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Segment-level Confusion Matrix:")
segment_cm_df = pd.DataFrame(
    segment_cm, index=class_names, columns=class_names
)
print(segment_cm_df)

In [ ]:
# Plot TIC-level confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=tic_cm, display_labels=class_names
)
disp.plot(ax=ax, values_format="d", xticks_rotation=45, cmap="viridis")
ax.set_title(f"{eval_split.upper()} Split - TIC-level Confusion Matrix", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("TIC-level Confusion Matrix:")
tic_cm_df = pd.DataFrame(
    tic_cm, index=class_names, columns=class_names
)
print(tic_cm_df)

## Step 9: Detailed Classification Report

In [ ]:
# Segment-level classification report
print("\n" + "="*60)
print("SEGMENT-LEVEL CLASSIFICATION REPORT")
print("="*60)
segment_report = classification_report(
    pred["labels"], segment_preds, target_names=class_names, zero_division=0
)
print(segment_report)

In [ ]:
# TIC-level classification report
print("\n" + "="*60)
print("TIC-LEVEL CLASSIFICATION REPORT")
print("="*60)
tic_report = classification_report(
    tic_agg["tic_true"], tic_agg["tic_pred"], target_names=class_names, zero_division=0
)
print(tic_report)

## Step 10: Summary and Export Results

In [ ]:
# Create summary dictionary
summary = {
    "checkpoint": str(model_file),
    "split": eval_split,
    "segment_metrics": {k: float(v) if isinstance(v, (int, np.integer, float, np.floating)) else v 
                        for k, v in segment_metrics.items()},
    "tic_metrics": {k: float(v) if isinstance(v, (int, np.integer, float, np.floating)) else v 
                   for k, v in tic_metrics.items()},
    "segment_confusion_matrix": segment_cm.tolist(),
    "tic_confusion_matrix": tic_cm.tolist(),
    "class_names": class_names,
}

print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(json.dumps(summary, indent=2, default=str))